In [ ]:
import os
os.system("pkill -f streamlit")
os.system("pkill -f pinggy")

!pip install -q -U PyMuPDF streamlit torch transformers accelerate bitsandbytes rank_bm25 scikit-learn numpy Pillow faiss-cpu langchain-text-splitters sentence-transformers docling streamlit-pdf-viewer
print("✅ Temiz kurulum tamamlandı!")


In [ ]:
%%writefile app.py
# -*- coding: utf-8 -*-
import os
import streamlit as st
import torch
import torch.nn.functional as F
import fitz  # PyMuPDF
import numpy as np
import gc
import re
import faiss
from PIL import Image
from threading import Thread
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel, BitsAndBytesConfig, TextIteratorStreamer
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter
from docling.document_converter import DocumentConverter
from streamlit_pdf_viewer import pdf_viewer

# Bellek fragmantasyonunu önlemek için PyTorch ayarı
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not os.path.exists("uploaded_pdfs"):
    os.makedirs("uploaded_pdfs")

# ==============================================================================
# 1. VEKTÖR MOTORUMUZ
# ==============================================================================
class CustomEmbedder:
    def __init__(self, model_name="BAAI/bge-m3", device="cuda"):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)

    def encode(self, sentences):
        if isinstance(sentences, str): sentences = [sentences]
        encoded_input = self.tokenizer(sentences, padding=True, truncation=True, max_length=512, return_tensors='pt').to(self.device)
        with torch.no_grad():
            model_output = self.model(**encoded_input)
        
        sentence_embs = model_output[0][:, 0]
        sentence_embs = F.normalize(sentence_embs, p=2, dim=1)
        return sentence_embs.cpu().numpy()

# ==============================================================================
# 2. MODEL YÜKLEME
# ==============================================================================
@st.cache_resource(show_spinner=False)
def load_models():
    model_id = "Qwen/Qwen2.5-14B-Instruct"
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=bnb_config, trust_remote_code=True)
    
    embedder = CustomEmbedder(device="cuda") 
    reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512, device="cuda")
    converter = DocumentConverter()
    
    return tokenizer, model, embedder, reranker, converter

# ==============================================================================
# 3. KUSURSUZ DOCLING PARÇALAMA
# ==============================================================================
def process_documents(uploaded_files, converter, chunk_size=1000, overlap=250):
    chunks = []
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    
    for file in uploaded_files:
        save_path = os.path.join("uploaded_pdfs", file.name)
        with open(save_path, "wb") as f:
            f.write(file.getbuffer())
            
        doc = fitz.open(save_path)
        for page_num, page in enumerate(doc):
            text = page.get_text("text") 
            if not text or not text.strip(): continue
            
            split_texts = text_splitter.split_text(text)
            for chunk_text in split_texts:
                if len(chunk_text) > 50:
                    chunks.append({
                        "pdf_path": save_path,
                        "filename": file.name,
                        "page_num": page_num + 1, 
                        "text": chunk_text
                    })
        doc.close()
    return chunks


# ==============================================================================
# BUTON CALLBACK FONKSİYONU (HIZLI VE HATASIZ ÇALIŞMASI İÇİN)
# ==============================================================================
def show_source_in_viewer(pdf_path, page_num, text, query):
    st.session_state.viewer_pdf_path = pdf_path
    st.session_state.viewer_page = page_num
    
    # Reranker modelini cache'den çek
    _, _, _, reranker, _ = load_models()
    
    doc = fitz.open(pdf_path)
    page = doc[page_num - 1]
    
    clean_text = text.replace("\n", " ")
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', clean_text) if len(s.strip()) > 15]
    if not sentences: sentences = [clean_text]
    target_sentences = sentences
    
    if reranker and query and len(sentences) > 1:
        try:
            scores = reranker.predict([[query, sent] for sent in sentences])
            top_indices = np.argsort(scores)[::-1][:2]
            target_sentences = [sentences[idx] for idx in top_indices]
        except:
            pass
            
    annotations = []
    
    # 3. YEPYENİ KUSURSUZ MİMARİ: 
    # n-gram (2'li kelime) araması sayfanın her yerinde alakasız kelimeleri boyuyordu.
    # Bunun yerine metnin orijinal PDF satırlarını (lines) alıyoruz.
    lines = [line.strip() for line in text.split('\n') if len(line.strip()) > 8]
    
    for line in lines:
        # Eğer bu orijinal PDF satırı, yapay zekanın seçtiği en iyi cümlenin bir parçasıysa:
        is_target = False
        for ts in target_sentences:
            if line in ts:
                is_target = True
                break
                
        if is_target:
            # Satır, PDF'in orijinal yapısından çekildiği için search_for() onu %100 doğru koordinatta bulur.
            for inst in page.search_for(line):
                annotations.append({
                    "page": page_num,
                    "x": inst.x0,
                    "y": inst.y0,
                    "width": (inst.x1 - inst.x0),
                    "height": (inst.y1 - inst.y0),
                    "color": "yellow"
                })
    doc.close()
    
    st.session_state.viewer_annotations = annotations
    st.session_state.viewer_key = f"viewer_{page_num}_{np.random.randint(1000)}"


# ==============================================================================
# 4. ARAYÜZ (CHAT & PDF VIEWER)
# ==============================================================================
st.set_page_config(page_title="Enterprise RAG Asistan", page_icon="📖", layout="wide")

if "messages" not in st.session_state: st.session_state.messages = []
if "chunks" not in st.session_state: st.session_state.chunks = []
if "faiss_index" not in st.session_state: st.session_state.faiss_index = None
if "bm25" not in st.session_state: st.session_state.bm25 = None
if "viewer_pdf_path" not in st.session_state: st.session_state.viewer_pdf_path = None
if "viewer_annotations" not in st.session_state: st.session_state.viewer_annotations = []
if "viewer_page" not in st.session_state: st.session_state.viewer_page = 1
if "viewer_key" not in st.session_state: st.session_state.viewer_key = "default_key"

with st.spinner("🧠 Profesyonel RAG ve Vizyon Modelleri Yükleniyor..."):
    llm_tokenizer, llm_model, embedder, reranker, converter = load_models()

with st.sidebar:
    st.title("📚 Belgeler")
    uploaded_files = st.file_uploader("PDF Yükle", type=["pdf"], accept_multiple_files=True)
    
    if st.button("📑 İşle ve Öğren", use_container_width=True):
        if uploaded_files:
            with st.spinner("Docling Mimarisiyle Hiyerarşik Parçalanıyor..."):
                chunks = process_documents(uploaded_files, converter)
                texts = [c["text"] for c in chunks]
                st.session_state.chunks = chunks
                
                with st.spinner("Vektör veritabanı oluşturuluyor..."):
                    embeddings = embedder.encode(texts)
                    dimension = embeddings.shape[1]
                    index = faiss.IndexFlatIP(dimension) 
                    index.add(embeddings)
                    st.session_state.faiss_index = index
                    st.session_state.bm25 = BM25Okapi([t.lower().split() for t in texts])
                    
                st.success("✅ Belgeler Hazır!")

    st.markdown("---")
    st.markdown("### ⚙️ Sohbet Ayarları")
    if st.button("🗑️ Sohbet Geçmişini Temizle", type="primary", use_container_width=True):
        st.session_state.messages = []
        st.session_state.viewer_pdf_path = None
        st.rerun()

# Layout: Sol Chat, Sağ PDF Viewer
col1, col2 = st.columns([6, 4])

with col1:
    st.title("📖 Enterprise RAG Asistanı")
    st.markdown("Etkileşimli PDF Okuyucu ve Docling Mimarisi.")

    for msg_idx, msg in enumerate(st.session_state.messages):
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])
            if "sources" in msg and msg["sources"]:
                cols = st.columns(min(len(msg["sources"]), 4))
                for i, src in enumerate(msg["sources"][:4]):
                    cols[i].button(
                        f"🔍 {src['filename']} (S:{src['page_num']})", 
                        key=f"btn_{msg_idx}_{i}",
                        on_click=show_source_in_viewer,
                        args=(src['pdf_path'], src['page_num'], src['text'], msg.get("query", ""))
                    )

    if prompt := st.chat_input("Soru sorun..."):
        if not st.session_state.chunks:
            st.warning("Önce belge yükleyin.")
            st.stop()

        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"): st.markdown(prompt)

        with st.spinner("Araştırılıyor..."):
            query_emb = embedder.encode([prompt])
            k = min(25, len(st.session_state.chunks))
            faiss_scores, faiss_indices = st.session_state.faiss_index.search(query_emb, k)
            vec_ranks = {idx: rank for rank, idx in enumerate(faiss_indices[0]) if idx != -1}
            
            bm25_scores = st.session_state.bm25.get_scores(prompt.lower().split())
            bm25_top_indices = np.argsort(bm25_scores)[::-1][:k]
            bm25_ranks = {idx: rank for rank, idx in enumerate(bm25_top_indices)}
            
            hybrid_pool_indices = set(list(vec_ranks.keys()) + list(bm25_ranks.keys()))
            rrf = {}
            for idx in hybrid_pool_indices:
                vec_rank = vec_ranks.get(idx, 60)
                bm25_rank = bm25_ranks.get(idx, 60)
                rrf[idx] = (1.0 / (60 + vec_rank)) + (1.0 / (60 + bm25_rank))
            
            top_hybrid_ids = sorted(rrf, key=rrf.get, reverse=True)[:k]
            
            cross_inp = [[prompt, st.session_state.chunks[i]["text"]] for i in top_hybrid_ids]
            rerank_scores = reranker.predict(cross_inp)
            
            final_k = min(8, len(top_hybrid_ids))
            reranked_pairs = sorted(zip(top_hybrid_ids, rerank_scores), key=lambda x: x[1], reverse=True)[:final_k]
            top_ids = [pair[0] for pair in reranked_pairs]
            
            retrieved_sources = [st.session_state.chunks[i] for i in top_ids]
            
            context_str = ""
            for idx, src in enumerate(retrieved_sources):
                context_str += f"\n[KAYNAK {idx+1}]\n{src['text']}\n"

        system_prompt = f"""<|im_start|>system
Sen profesyonel bir yapay zeka asistanısın. Lütfen soruları yanıtlarken doğal, resmi ve akıcı bir Türkçe kullan.

KURALLAR:
1. Sadece aşağıdaki belgelerde yer alan bilgileri kullanarak yanıt ver.
2. Verdiğin bilgilerin sonuna mutlaka [KAYNAK X] şeklinde referans ekle.
3. Sorunun cevabı belgelerde yoksa, "Bu bilgi belgelerde bulunmuyor" de.

BELGELER:
{context_str}
<|im_end|>
"""
        
        prompt_parts = [system_prompt]
        recent_history = st.session_state.messages[-5:-1]
        for msg in recent_history:
            prompt_parts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n")
        
        prompt_parts.append(f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n")
        
        inputs = llm_tokenizer("".join(prompt_parts), return_tensors="pt").to("cuda")
        streamer = TextIteratorStreamer(llm_tokenizer, timeout=40., skip_prompt=True, skip_special_tokens=True)
        
        generation_kwargs = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=4096,
            temperature=0.4,
            top_p=0.9,                
            repetition_penalty=1.0,
            pad_token_id=llm_tokenizer.eos_token_id
        )

        with st.chat_message("assistant"):
            thread = Thread(target=llm_model.generate, kwargs=generation_kwargs)
            thread.start()
            full_response = st.write_stream(streamer)

        st.session_state.messages.append({"role": "assistant", "content": full_response, "sources": retrieved_sources, "query": prompt})
        
        # YENİ ÖZELLİK: Asistan cevap verdiğinde en iyi kaynağı otomatik olarak sağda aç
        if retrieved_sources:
            show_source_in_viewer(retrieved_sources[0]['pdf_path'], retrieved_sources[0]['page_num'], retrieved_sources[0]['text'], prompt)
            st.rerun()

with col2:
    if st.session_state.viewer_pdf_path is not None and os.path.exists(st.session_state.viewer_pdf_path):
        st.markdown("### 📄 Hedef Kaynak")
        with open(st.session_state.viewer_pdf_path, "rb") as f:
            pdf_bytes = f.read()
            
        pdf_viewer(
            input=pdf_bytes,
            width=700,
            annotations=st.session_state.viewer_annotations,
            pages_to_render=[st.session_state.viewer_page],
            key=st.session_state.viewer_key
        )
    else:
        st.markdown("### 📄 Hedef Kaynak")
        st.info("Bir cevaptaki kaynağı görmek için 🔍 butonlarına tıklayın.")


In [ ]:
import os
import time
import subprocess
import re

print("⏳ Tünel başlatılıyor, lütfen bekleyin...")

# Eski işlemleri tamamen temizle
os.system("pkill -f streamlit")
os.system("pkill -f pinggy")
time.sleep(2)

# Streamlit uygulamasını başlat
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"])
time.sleep(4) 

# Log dosyasını sıfırla
os.system("rm -f pinggy_log.txt")

# Pinggy tünelini arka planda başlat
cmd = "ssh -p 443 -R0:localhost:8501 -o StrictHostKeyChecking=no -o ServerAliveInterval=30 a.pinggy.io > pinggy_log.txt 2>&1 &"
os.system(cmd)

# Pinggy'nin URL'yi üretmesi için kontrol döngüsü
url_found = False
for _ in range(15): 
    time.sleep(1)
    if os.path.exists("pinggy_log.txt"):
        with open("pinggy_log.txt", "r") as f:
            log_content = f.read()
            urls = re.findall(r"https://[a-zA-Z0-9-]+\.a\.free\.pinggy\.link", log_content)
            if urls:
                print("\n" + "="*65)
                print("🚀 SİSTEM HAZIR! ARAYÜZE GİRMEK İÇİN TIKLAYIN:")
                print(f"👉 {urls[0]}")
                print("="*65)
                url_found = True
                break

if not url_found:
    print("\n⚠️ Tünel oluşturulamadı. Log detayları:")
    os.system("cat pinggy_log.txt")
